In [1]:
bibliography_file = '../../../data/biblio-v5-table.ods'

In [2]:
import polars as pl

In [3]:
df_raw = pl.read_ods(bibliography_file)

In [4]:
# Process the 'pages' column to split it into 'start' and 'end' pages by "--"

df = df_raw.select([
    "bibkey",
    "title",
    "author",
    "date",
    "journal",
    "volume",
    "number",
    "pages"
]).with_columns([
    pl.col('pages')
    .str.strip_chars()
    .str.split('--')
    .alias('pages_split')
]).with_columns([
    pl.col('pages_split').list.get(0, null_on_oob=True).alias('start'),
    pl.col('pages_split').list.get(1, null_on_oob=True).alias('end')
]).drop('pages_split').with_columns([
    pl.col('start').str.replace_all(r'[^0-9]', '').cast(pl.Int32, strict=False).alias('start_int'),
])

In [5]:
sorted = df.sort('start_int', nulls_last=True).sort("number", descending=True).sort('volume', descending=True).sort('date', descending=True)

In [6]:

nous = sorted.filter(pl.col('journal') == 'No{\\^u}s')

In [7]:
#journal_items.sort('date').sort("volume").sort("number").sort('start')

apq = sorted.filter(pl.col('journal') == 'American Philosophical Quarterly')

In [8]:
apq

bibkey,title,author,date,journal,volume,number,pages,start,end,start_int
str,str,str,i64,str,i64,str,str,str,str,i32
"""duncan_m:2021""","""Animalism is Either False or U…","""Duncan, Matt""",2021,"""American Philosophical Quarter…",58,"""2""","""187--200""","""187""","""200""",187
"""thornton_ak:2019""","""Disembodied Animals""","""Thornton, Allison Krile""",2019,"""American Philosophical Quarter…",56,"""2""","""203--217""","""203""","""217""",203
"""wedgwood_r:2019""","""Moral Disagreement and Inexcus…","""Wedgwood, Ralph""",2019,"""American Philosophical Quarter…",56,"""1""","""97--108""","""97""","""108""",97
"""cowling_s-cray:2017""","""How to Be Omnipresent""","""Cowling, Sam and Cray, Wesley …",2017,"""American Philosophical Quarter…",54,"""3""","""223--234""","""223""","""234""",223
"""bourget:2017""","""Representationalism and Sensor…","""Bourget, David""",2017,"""American Philosophical Quarter…",54,"""3""","""251--268""","""251""","""268""",251
…,…,…,…,…,…,…,…,…,…,…
"""chisholm:1964""","""The Ethics of Requirement""","""Chisholm, Roderick M.""",1964,"""American Philosophical Quarter…",1,null,"""147--153""","""147""","""153""",147
"""thalberg:1964a""","""Emotion and Thought""","""Thalberg, Irving""",1964,"""American Philosophical Quarter…",1,null,null,null,null,null
"""spiegelberg:1964""","""Toward a Phenomenology of Expe…","""Spiegelberg, Herbert""",1964,"""American Philosophical Quarter…",1,"""4""","""325--332""","""325""","""332""",325


In [9]:
import sys

sys.path.append("../..")

In [10]:
from ref_pipe.models import HTMLIssue, HTMLVolume, HTMLYear, THTMLCollapsible  # type: ignore


def build_collapsible(df: pl.DataFrame) -> THTMLCollapsible:
    years = []
    for year_val, year_group in df.group_by("date", maintain_order=True):
        year_str = " ".join(map(str, year_val))

        volumes = year_group.select("volume").unique().drop_nulls().to_series().to_list()
        has_multiple_volumes = len(volumes) > 1

        bibs_without_volume = year_group.filter(pl.col("volume").is_null())["bibkey"].to_list()

        if not has_multiple_volumes:
            # Single or no volume: flatten to HTMLYear
            year_bibkeys = year_group["bibkey"].to_list()
            name = year_str
            if len(volumes) == 1:
                name += f" :: vol. {volumes[0]}"
            years.append(HTMLYear(name=name, contents=tuple(year_bibkeys)))
        else:
            # Multiple volumes
            year_contents = [*bibs_without_volume]
            volume_names_g = (
                f"{v}" for v in year_group["volume"].unique().drop_nulls().to_list()
            )
            volume_names = ", ".join(volume_names_g)
            year_name = f"{year_str} :: vol. {volume_names}"

            for volume_val, vol_group in year_group.group_by("volume", maintain_order=True):
                volume_str = " ".join(map(str, volume_val))

                volume_bibkeys_without_issue = vol_group.filter(pl.col("number").is_null().or_(pl.col("number") == ""))["bibkey"].to_list()
                volume_contents = [*volume_bibkeys_without_issue]

                issues = vol_group.select("number").unique().drop_nulls().to_series().to_list()
                for issue_val in issues:
                    issue_str = " ".join(map(str, issue_val))
                    issue_name = f"issue {issue_str}"

                    issue_bibkeys = vol_group.filter(pl.col("number") == issue_val)["bibkey"].to_list()
                    volume_contents.append(
                        HTMLIssue(name=str(issue_name), contents=tuple(issue_bibkeys))
                    )

                year_contents.append(HTMLVolume(name=volume_str, contents=tuple(volume_contents)))

            years.append(HTMLYear(name=year_name, contents=tuple(year_contents)))

    return tuple(years)


In [11]:
struct = build_collapsible(apq)

In [12]:
from pprint import pprint
pprint(struct)

(HTMLYear(name='2021 :: vol. 58', contents=('duncan_m:2021',)),
 HTMLYear(name='2019 :: vol. 56',
          contents=('thornton_ak:2019', 'wedgwood_r:2019')),
 HTMLYear(name='2017 :: vol. 54',
          contents=('cowling_s-cray:2017',
                    'bourget:2017',
                    'brunero:2017',
                    'gardiner_g:2017',
                    'miller_k:2017',
                    'shiller:2017',
                    'smith_njj:2017b',
                    'fuqua:2017',
                    'dunn_j-ahlstromvij:2017',
                    'hales_sd:2017',
                    'rescher:2017',
                    'bozickovic:2017',
                    'menges_al:2017a',
                    'turri-buckwalter:2017',
                    'forrest_p:2017b',
                    'fresco-etal:2017',
                    'cornell_dm:2017',
                    'sinhababu:2017a')),
 HTMLYear(name='2016 :: vol. 53',
          contents=('wyatt_j-lynch:2016',
                    'kind:201

In [13]:
struct_a = [x for x in struct if x.name == "1982"]

In [14]:
for x in struct:
    if isinstance(x, HTMLYear):
        #print(f"  Year {x.name}")
        for y in x.contents:
            if isinstance(y, HTMLVolume):
                for z in y.contents:
                    #print(f"      Volume {y.name}")
                    if isinstance(z, HTMLIssue):
                        for a in z.contents:
                            #print(f"          Issue {z.name}")
                            print(a) 
            else:
                print(y)
    else:
        print(x)

duncan_m:2021
thornton_ak:2019
wedgwood_r:2019
cowling_s-cray:2017
bourget:2017
brunero:2017
gardiner_g:2017
miller_k:2017
shiller:2017
smith_njj:2017b
fuqua:2017
dunn_j-ahlstromvij:2017
hales_sd:2017
rescher:2017
bozickovic:2017
menges_al:2017a
turri-buckwalter:2017
forrest_p:2017b
fresco-etal:2017
cornell_dm:2017
sinhababu:2017a
wyatt_j-lynch:2016
kind:2016d
vallier:2016
roche_m:2016
thompson_n:2016a
simion-kelp:2016
dorsey_je:2016
fazekas_k:2016
carter_ja-palermos:2016
mckenna_r:2016
lemos_nm:2016
capes_ja:2016a
manela:2016
masto:2016
mcconnell_d:2016
farkas_k:2016
frances_b:2016b
kersten_l-wilson:2016
neta:2016a
rupert:2016
schroeter_l-schroeter:2016
genone:2016
kelp:2016
oleary_s:2016
menzel_c:2016
weintraub_r:2016
loehrer-sehon:2016
mcrae_e:2016
thompson_n:2015
molyneux:2015a
locke_du:2015
bayer_b:2015
rieger_a:2015
werner_pj:2015
watson_la:2015
wieland_jw:2015
arpaly-schroeder:2015
fernandez_pa:2015
ford_a:2015
paul_sk:2015a
schroeder_m:2015i
sliwa:2015a
smith_ma:2015a
mumford_s

In [15]:
from ref_pipe.models import TBibkey, Bibliography
from aletk.utils import get_logger

lgr = get_logger(__name__)

year_format = """
<details class="collapsible-level-1">
    <summary>{name}</summary>
    <hr>

    {contents}

</details>
"""

volume_format = """
<details class="collapsible-level-2">
    <summary>{name}</summary>
    <hr>

    {contents}
</details>
"""

issue_format = """
<details class="collapsible-level-3">
    <summary>{name}</summary>
    <hr>

    {contents}
</details>
"""

def _get_bibkey_html(bibkey: TBibkey, bibliography: Bibliography) -> str:
    index = bibliography.bibkey_index_dict.get(bibkey, None)
    if index is None:
        lgr.warning(f"Bibkey {bibkey} not found in bibliography.")
        return ""
    if not isinstance(index, int) or index < 0:
        lgr.warning(f"Bibkey index {index} is not valid.")
        return "" 
    
    try:
        html = bibliography.content[index]
        assert isinstance(html, str)

    except Exception as e:
        lgr.warning(f"Error getting HTML for bibkey '{bibkey}': {e}")
        return ""

    return html

In [16]:
bibdiv_dict = {
    "hello:1920": "<p id=\"hello:1920\">Hello</p>",
    "hello:1921": "<p id=\"hello:1921\">Hello</p>",
    "hello:1922": "<p id=\"hello:1922\">Hello</p>",
    "hello:1923": "<p id=\"hello:1923\">Hello</p>",
    "hello:1924": "<p id=\"hello:1924\">Hello</p>",
    "hello:1925": "<p id=\"hello:1925\">Hello</p>",
    "hello:1926": "<p id=\"hello:1926\">Hello</p>",
    "hello:1927": "<p id=\"hello:1927\">Hello</p>",
    "hello:1928": "<p id=\"hello:1928\">Hello</p>",
    "hello:1929": "<p id=\"hello:1929\">Hello</p>",
    "hello:1930": "<p id=\"hello:1930\">Hello</p>",
    "byebye:1930": "<p id=\"byebye:1930\">Goodbye</p>",
    "byebye:1931": "<p id=\"byebye:1931\">Goodbye</p>",
    "byebye:1932": "<p id=\"byebye:1932\">Goodbye</p>",
    "byebye:1933": "<p id=\"byebye:1933\">Goodbye</p>",
    "byebye:1934": "<p id=\"byebye:1934\">Goodbye</p>",
    "byebye:1935": "<p id=\"byebye:1935\">Goodbye</p>",
    "byebye:1936": "<p id=\"byebye:1936\">Goodbye</p>",
    "byebye:1937": "<p id=\"byebye:1937\">Goodbye</p>",
    "byebye:1938": "<p id=\"byebye:1938\">Goodbye</p>",
    "byebye:1939": "<p id=\"byebye:1939\">Goodbye</p>",
    "byebye:1940": "<p id=\"byebye:1940\">Goodbye</p>",
    "duncan_m:2021": "<p id=\"duncan_m:2021\">Duncan</p>",
    "thornton_ak:2019": "<p id=\"duncan_m:2021\">Duncan</p>",
    "wedgwood_r:2019": "<p id=\"duncan_m:2021\">Duncan</p>",
    "cowling_s-cray:2017": "<p id=\"duncan_m:2021\">Duncan</p>",
    "bourget:2017": "<p id=\"duncan_m:2021\">Duncan</p>",
    "brunero:2017": "<p id=\"duncan_m:2021\">Duncan</p>",
    "gardiner_g:2017": "<p id=\"duncan_m:2021\">Duncan</p>",
    "miller_k:2017": "<p id=\"duncan_m:2021\">Duncan</p>",
    "shiller:2017": "<p id=\"duncan_m:2021\">Duncan</p>",
    "smith_njj:2017b": "<p id=\"duncan_m:2021\">Duncan</p>",
    "fuqua:2017": "<p id=\"duncan_m:2021\">Duncan</p>",
    "dunn_j-ahlstromvij:2017": "<p id=\"duncan_m:2021\">Duncan</p>",
    "hales_sd:2017": "<p id=\"duncan_m:2021\">Duncan</p>",
    "rescher:2017": "<p id=\"duncan_m:2021\">Duncan</p>",
    "bozickovic:2017": "<p id=\"duncan_m:2021\">Duncan</p>",
    "menges_al:2017a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "turri-buckwalter:2017": "<p id=\"duncan_m:2021\">Duncan</p>",
    "forrest_p:2017b": "<p id=\"duncan_m:2021\">Duncan</p>",
    "fresco-etal:2017": "<p id=\"duncan_m:2021\">Duncan</p>",
    "cornell_dm:2017": "<p id=\"duncan_m:2021\">Duncan</p>",
    "sinhababu:2017a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "wyatt_j-lynch:2016": "<p id=\"duncan_m:2021\">Duncan</p>",
    "kind:2016d": "<p id=\"duncan_m:2021\">Duncan</p>",
    "vallier:2016": "<p id=\"duncan_m:2021\">Duncan</p>",
    "roche_m:2016": "<p id=\"duncan_m:2021\">Duncan</p>",
    "thompson_n:2016a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "simion-kelp:2016": "<p id=\"duncan_m:2021\">Duncan</p>",
    "dorsey_je:2016": "<p id=\"duncan_m:2021\">Duncan</p>",
    "fazekas_k:2016": "<p id=\"duncan_m:2021\">Duncan</p>",
    "carter_ja-palermos:2016": "<p id=\"duncan_m:2021\">Duncan</p>",
    "mckenna_r:2016": "<p id=\"duncan_m:2021\">Duncan</p>",
    "lemos_nm:2016": "<p id=\"duncan_m:2021\">Duncan</p>",
    "capes_ja:2016a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "manela:2016": "<p id=\"duncan_m:2021\">Duncan</p>",
    "masto:2016": "<p id=\"duncan_m:2021\">Duncan</p>",
    "mcconnell_d:2016": "<p id=\"duncan_m:2021\">Duncan</p>",
    "farkas_k:2016": "<p id=\"duncan_m:2021\">Duncan</p>",
    "frances_b:2016b": "<p id=\"duncan_m:2021\">Duncan</p>",
    "kersten_l-wilson:2016": "<p id=\"duncan_m:2021\">Duncan</p>",
    "neta:2016a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "rupert:2016": "<p id=\"duncan_m:2021\">Duncan</p>",
    "schroeter_l-schroeter:2016": "<p id=\"duncan_m:2021\">Duncan</p>",
    "genone:2016": "<p id=\"duncan_m:2021\">Duncan</p>",
    "kelp:2016": "<p id=\"duncan_m:2021\">Duncan</p>",
    "oleary_s:2016": "<p id=\"duncan_m:2021\">Duncan</p>",
    "menzel_c:2016": "<p id=\"duncan_m:2021\">Duncan</p>",
    "weintraub_r:2016": "<p id=\"duncan_m:2021\">Duncan</p>",
    "loehrer-sehon:2016": "<p id=\"duncan_m:2021\">Duncan</p>",
    "mcrae_e:2016": "<p id=\"duncan_m:2021\">Duncan</p>",
    "thompson_n:2015": "<p id=\"duncan_m:2021\">Duncan</p>",
    "arpaly-schroeder:2015": "<p id=\"duncan_m:2021\">Duncan</p>",
    "fernandez_pa:2015": "<p id=\"duncan_m:2021\">Duncan</p>",
    "ford_a:2015": "<p id=\"duncan_m:2021\">Duncan</p>",
    "paul_sk:2015a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "schroeder_m:2015i": "<p id=\"duncan_m:2021\">Duncan</p>",
    "sliwa:2015a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "smith_ma:2015a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "molyneux:2015a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "locke_du:2015": "<p id=\"duncan_m:2021\">Duncan</p>",
    "bayer_b:2015": "<p id=\"duncan_m:2021\">Duncan</p>",
    "rieger_a:2015": "<p id=\"duncan_m:2021\">Duncan</p>",
    "werner_pj:2015": "<p id=\"duncan_m:2021\">Duncan</p>",
    "watson_la:2015": "<p id=\"duncan_m:2021\">Duncan</p>",
    "wieland_jw:2015": "<p id=\"duncan_m:2021\">Duncan</p>",
    "varga_s:2015a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "koksvik:2015": "<p id=\"duncan_m:2021\">Duncan</p>",
    "sorensen_ra:2015b": "<p id=\"duncan_m:2021\">Duncan</p>",
    "baron_s:2015b": "<p id=\"duncan_m:2021\">Duncan</p>",
    "valaris-michael:2015": "<p id=\"duncan_m:2021\">Duncan</p>",
    "goldberg_sc:2015e": "<p id=\"duncan_m:2021\">Duncan</p>",
    "fischer_rw:2015": "<p id=\"duncan_m:2021\">Duncan</p>",
    "mumford_s-anjum:2015": "<p id=\"duncan_m:2021\">Duncan</p>",
    "bandyopadhyay_p-etal:2015": "<p id=\"duncan_m:2021\">Duncan</p>",
    "baron_s-miller:2015": "<p id=\"duncan_m:2021\">Duncan</p>",
    "greco_d:2015e": "<p id=\"duncan_m:2021\">Duncan</p>",
    "lewtas_p:2015": "<p id=\"duncan_m:2021\">Duncan</p>",
    "peels:2015a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "sorell:2015": "<p id=\"duncan_m:2021\">Duncan</p>",
    "thomasson:2014a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "gracia_j:2014": "<p id=\"duncan_m:2021\">Duncan</p>",
    "cumpa:2014b": "<p id=\"duncan_m:2021\">Duncan</p>",
    "bueno_o:2014": "<p id=\"duncan_m:2021\">Duncan</p>",
    "heil_j:2014": "<p id=\"duncan_m:2021\">Duncan</p>",
    "oderberg:2014": "<p id=\"duncan_m:2021\">Duncan</p>",
    "williams_ne:2014": "<p id=\"duncan_m:2021\">Duncan</p>",
    "mcdaniel_b:2014": "<p id=\"duncan_m:2021\">Duncan</p>",
    "reiland:2014": "<p id=\"duncan_m:2021\">Duncan</p>",
    "bedke:2014a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "dunn_j:2014": "<p id=\"duncan_m:2021\">Duncan</p>",
    "nanay:2014a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "kelp:2014a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "swinburne:2014": "<p id=\"duncan_m:2021\">Duncan</p>",
    "ware_o:2014": "<p id=\"duncan_m:2021\">Duncan</p>",
    "cholbi:2014a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "wringe_b:2014": "<p id=\"duncan_m:2021\">Duncan</p>",
    "pickavance:2014": "<p id=\"duncan_m:2021\">Duncan</p>",
    "declercq_r-etal:2014": "<p id=\"duncan_m:2021\">Duncan</p>",
    "shemmer:2014": "<p id=\"duncan_m:2021\">Duncan</p>",
    "mccammon:2014": "<p id=\"duncan_m:2021\">Duncan</p>",
    "graber_a:2014": "<p id=\"duncan_m:2021\">Duncan</p>",
    "foddy:2014": "<p id=\"duncan_m:2021\">Duncan</p>",
    "garcia_rk:2014": "<p id=\"duncan_m:2021\">Duncan</p>",
    "carter_ja-gordon:2014": "<p id=\"duncan_m:2021\">Duncan</p>",
    "shepherd_j:2014": "<p id=\"duncan_m:2021\">Duncan</p>",
    "green_a:2014": "<p id=\"duncan_m:2021\">Duncan</p>",
    "sauchelli:2014": "<p id=\"duncan_m:2021\">Duncan</p>",
    "gorin_m:2014": "<p id=\"duncan_m:2021\">Duncan</p>",
    "bramble:2014": "<p id=\"duncan_m:2021\">Duncan</p>",
    "gehrman:2014": "<p id=\"duncan_m:2021\">Duncan</p>",
    "anders_pc-etal:2014": "<p id=\"duncan_m:2021\">Duncan</p>",
    "schroer_r:2013b": "<p id=\"duncan_m:2021\">Duncan</p>",
    "howardsnyder_d:2013": "<p id=\"duncan_m:2021\">Duncan</p>",
    "bonjour:2013": "<p id=\"duncan_m:2021\">Duncan</p>",
    "cox_d:2013": "<p id=\"duncan_m:2021\">Duncan</p>",
    "kraay_k:2013": "<p id=\"duncan_m:2021\">Duncan</p>",
    "ballantyne-thurow:2013": "<p id=\"duncan_m:2021\">Duncan</p>",
    "borge:2013": "<p id=\"duncan_m:2021\">Duncan</p>",
    "glock_hj:2013a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "carruthers:2013": "<p id=\"duncan_m:2021\">Duncan</p>",
    "allen_c:2013": "<p id=\"duncan_m:2021\">Duncan</p>",
    "moyalsharrock:2013": "<p id=\"duncan_m:2021\">Duncan</p>",
    "hutto:2013": "<p id=\"duncan_m:2021\">Duncan</p>",
    "rowlands:2013": "<p id=\"duncan_m:2021\">Duncan</p>",
    "medina_j:2013a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "orozco:2013": "<p id=\"duncan_m:2021\">Duncan</p>",
    "mckinnon_r:2013": "<p id=\"duncan_m:2021\">Duncan</p>",
    "maguidhir:2013a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "jech_a:2013": "<p id=\"duncan_m:2021\">Duncan</p>",
    "rasmussen_j:2013": "<p id=\"duncan_m:2021\">Duncan</p>",
    "taylor_e:2013a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "raven_mj:2013a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "hershenov:2013": "<p id=\"duncan_m:2021\">Duncan</p>",
    "kriegel_u:2013d": "<p id=\"duncan_m:2021\">Duncan</p>",
    "abell_c:2013": "<p id=\"duncan_m:2021\">Duncan</p>",
    "tropman:2013": "<p id=\"duncan_m:2021\">Duncan</p>",
    "beillard:2013": "<p id=\"duncan_m:2021\">Duncan</p>",
    "gregory_a:2013": "<p id=\"duncan_m:2021\">Duncan</p>",
    "shaffer_mj:2013": "<p id=\"duncan_m:2021\">Duncan</p>",
    "curzer:2013": "<p id=\"duncan_m:2021\">Duncan</p>",
    "goldstein_l:2013b": "<p id=\"duncan_m:2021\">Duncan</p>",
    "klocksiem:2012": "<p id=\"duncan_m:2021\">Duncan</p>",
    "matthews_st-kennett:2012": "<p id=\"duncan_m:2021\">Duncan</p>",
    "blatti:2012a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "mccarty_r:2012": "<p id=\"duncan_m:2021\">Duncan</p>",
    "smith_njj:2012b": "<p id=\"duncan_m:2021\">Duncan</p>",
    "nagasawa_y:2012a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "mele_a:2012b": "<p id=\"duncan_m:2021\">Duncan</p>",
    "axtell_g-olson:2012": "<p id=\"duncan_m:2021\">Duncan</p>",
    "bell_mac:2012": "<p id=\"duncan_m:2021\">Duncan</p>",
    "tiehen:2012": "<p id=\"duncan_m:2021\">Duncan</p>",
    "comesana:2012": "<p id=\"duncan_m:2021\">Duncan</p>",
    "melanson:2012": "<p id=\"duncan_m:2021\">Duncan</p>",
    "peijnenburg-atkinson:2012": "<p id=\"duncan_m:2021\">Duncan</p>",
    "long_jo:2012": "<p id=\"duncan_m:2021\">Duncan</p>",
    "hinchman:2012": "<p id=\"duncan_m:2021\">Duncan</p>",
    "coates_a:2012": "<p id=\"duncan_m:2021\">Duncan</p>",
    "ross_pw:2012": "<p id=\"duncan_m:2021\">Duncan</p>",
    "clarke_rog:2012": "<p id=\"duncan_m:2021\">Duncan</p>",
    "nuccetelli-seay:2012a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "giberman_d:2012": "<p id=\"duncan_m:2021\">Duncan</p>",
    "byron_m:2012": "<p id=\"duncan_m:2021\">Duncan</p>",
    "armourgarb:2012a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "greco_j:2012": "<p id=\"duncan_m:2021\">Duncan</p>",
    "gibb:2012": "<p id=\"duncan_m:2021\">Duncan</p>",
    "sneddon:2012": "<p id=\"duncan_m:2021\">Duncan</p>",
    "mcaleer_s:2012": "<p id=\"duncan_m:2021\">Duncan</p>",
    "williston:2012": "<p id=\"duncan_m:2021\">Duncan</p>",
    "strandberg_c:2012": "<p id=\"duncan_m:2021\">Duncan</p>",
    "rajczi:2011": "<p id=\"duncan_m:2021\">Duncan</p>",
    "rice_r:2011": "<p id=\"duncan_m:2021\">Duncan</p>",
    "aranyosi_i:2011": "<p id=\"duncan_m:2021\">Duncan</p>",
    "kraay_k:2011": "<p id=\"duncan_m:2021\">Duncan</p>",
    "morris_k:2011a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "alm:2011": "<p id=\"duncan_m:2021\">Duncan</p>",
    "sorensen_ra:2011d": "<p id=\"duncan_m:2021\">Duncan</p>",
    "parsons_c:2011": "<p id=\"duncan_m:2021\">Duncan</p>",
    "fara_dg:2011a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "massey_gj:2011": "<p id=\"duncan_m:2021\">Duncan</p>",
    "elgin_cz:2011a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "follesdal_d:2011": "<p id=\"duncan_m:2021\">Duncan</p>",
    "harman_gh:2011a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "ricketts:2011": "<p id=\"duncan_m:2021\">Duncan</p>",
    "george_a:2011a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "dennett:2011": "<p id=\"duncan_m:2021\">Duncan</p>",
    "hacker_pms:2011a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "kenny_a:2011": "<p id=\"duncan_m:2021\">Duncan</p>",
    "searle_jr:2011b": "<p id=\"duncan_m:2021\">Duncan</p>",
    "hutto:2011": "<p id=\"duncan_m:2021\">Duncan</p>",
    "prinz_jj:2011a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "baker_r:2011": "<p id=\"duncan_m:2021\">Duncan</p>",
    "papineau:2011": "<p id=\"duncan_m:2021\">Duncan</p>",
    "flanagan_o:2011a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "williams_me:2011": "<p id=\"duncan_m:2021\">Duncan</p>",
    "zangwill_n:2011": "<p id=\"duncan_m:2021\">Duncan</p>",
    "orlandi:2011": "<p id=\"duncan_m:2021\">Duncan</p>",
    "hazlett_a-maguidhir:2011": "<p id=\"duncan_m:2021\">Duncan</p>",
    "morris_je:2011": "<p id=\"duncan_m:2021\">Duncan</p>",
    "cogburn-silcox:2011": "<p id=\"duncan_m:2021\">Duncan</p>",
    "arpaly:2011": "<p id=\"duncan_m:2021\">Duncan</p>",
    "kearns_s:2011a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "schnieder_b:2010b": "<p id=\"duncan_m:2021\">Duncan</p>",
    "claytoncoleman:2010": "<p id=\"duncan_m:2021\">Duncan</p>",
    "lepoidevin:2010": "<p id=\"duncan_m:2021\">Duncan</p>",
    "miller_k:2010": "<p id=\"duncan_m:2021\">Duncan</p>",
    "iacona:2010a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "debus_d:2010": "<p id=\"duncan_m:2021\">Duncan</p>",
    "smith_d:2010": "<p id=\"duncan_m:2021\">Duncan</p>",
    "goldman_ah:2009a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "gert_j:2009a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "cling:2009": "<p id=\"duncan_m:2021\">Duncan</p>",
    "douven:2009": "<p id=\"duncan_m:2021\">Duncan</p>",
    "littlejohn_c:2009": "<p id=\"duncan_m:2021\">Duncan</p>",
    "kim_s:2009": "<p id=\"duncan_m:2021\">Duncan</p>",
    "predelli:2009": "<p id=\"duncan_m:2021\">Duncan</p>",
    "brown_sr:2009": "<p id=\"duncan_m:2021\">Duncan</p>",
    "benovsky:2009": "<p id=\"duncan_m:2021\">Duncan</p>",
    "harrington_j:2009": "<p id=\"duncan_m:2021\">Duncan</p>",
    "pelczar:2009": "<p id=\"duncan_m:2021\">Duncan</p>",
    "miller_k:2009a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "espinoza_n:2009": "<p id=\"duncan_m:2021\">Duncan</p>",
    "helm_bw:2009b": "<p id=\"duncan_m:2021\">Duncan</p>",
    "bradley_dj:2009": "<p id=\"duncan_m:2021\">Duncan</p>",
    "vosgerau-etal:2008": "<p id=\"duncan_m:2021\">Duncan</p>",
    "russell_dc:2008": "<p id=\"duncan_m:2021\">Duncan</p>",
    "walker_mt:2008": "<p id=\"duncan_m:2021\">Duncan</p>",
    "goldstein_l-cave:2008": "<p id=\"duncan_m:2021\">Duncan</p>",
    "brynjarsdottir:2008": "<p id=\"duncan_m:2021\">Duncan</p>",
    "naylor_a:2008": "<p id=\"duncan_m:2021\">Duncan</p>",
    "legg_c:2008": "<p id=\"duncan_m:2021\">Duncan</p>",
    "lycan:2008a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "bedke:2008": "<p id=\"duncan_m:2021\">Duncan</p>",
    "rauti:2008": "<p id=\"duncan_m:2021\">Duncan</p>",
    "harold_j:2008": "<p id=\"duncan_m:2021\">Duncan</p>",
    "westphal_j:2008": "<p id=\"duncan_m:2021\">Duncan</p>",
    "goodman_le:2008": "<p id=\"duncan_m:2021\">Duncan</p>",
    "mele_a:2008": "<p id=\"duncan_m:2021\">Duncan</p>",
    "weidemann_h:2008": "<p id=\"duncan_m:2021\">Duncan</p>",
    "goldberg_sc:2008": "<p id=\"duncan_m:2021\">Duncan</p>",
    "rescher:2008b": "<p id=\"duncan_m:2021\">Duncan</p>",
    "gruenbaum_a:2008": "<p id=\"duncan_m:2021\">Duncan</p>",
    "hassoun:2008": "<p id=\"duncan_m:2021\">Duncan</p>",
    "pettit_g:2008": "<p id=\"duncan_m:2021\">Duncan</p>",
    "boran:2008": "<p id=\"duncan_m:2021\">Duncan</p>",
    "baumann_p:2008a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "collins_r:2008": "<p id=\"duncan_m:2021\">Duncan</p>",
    "adler_je-vasiliou:2008": "<p id=\"duncan_m:2021\">Duncan</p>",
    "predelli:2008c": "<p id=\"duncan_m:2021\">Duncan</p>",
    "jenkins_cs-nolan:2008a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "merli:2007": "<p id=\"duncan_m:2021\">Duncan</p>",
    "brogaard:2007d": "<p id=\"duncan_m:2021\">Duncan</p>",
    "gerken:2007": "<p id=\"duncan_m:2021\">Duncan</p>",
    "olsson_ej:2007": "<p id=\"duncan_m:2021\">Duncan</p>",
    "kulvicki:2007a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "calder_t:2007": "<p id=\"duncan_m:2021\">Duncan</p>",
    "scott_mi-stevens:2007": "<p id=\"duncan_m:2021\">Duncan</p>",
    "hill_dj:2007": "<p id=\"duncan_m:2021\">Duncan</p>",
    "fiocco:2007a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "weiss_b:2007b": "<p id=\"duncan_m:2021\">Duncan</p>",
    "wisnewski_jj-jacoby:2007": "<p id=\"duncan_m:2021\">Duncan</p>",
    "verbeek_b:2007": "<p id=\"duncan_m:2021\">Duncan</p>",
    "jenkins_cs:2007b": "<p id=\"duncan_m:2021\">Duncan</p>",
    "brady_m:2007": "<p id=\"duncan_m:2021\">Duncan</p>",
    "montmarquet_ja:2007a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "pritchard_d:2007b": "<p id=\"duncan_m:2021\">Duncan</p>",
    "elder_c:2007a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "ernst_z:2007a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "timpe:2007": "<p id=\"duncan_m:2021\">Duncan</p>",
    "saka:2007b": "<p id=\"duncan_m:2021\">Duncan</p>",
    "hershenov:2007": "<p id=\"duncan_m:2021\">Duncan</p>",
    "varzi:2007a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "montminy:2007a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "rajczi:2007": "<p id=\"duncan_m:2021\">Duncan</p>",
    "portmore:2007a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "bigelow-pargetter:2007": "<p id=\"duncan_m:2021\">Duncan</p>",
    "schueler_gf:2007": "<p id=\"duncan_m:2021\">Duncan</p>",
    "draper_k:2007": "<p id=\"duncan_m:2021\">Duncan</p>",
    "andreou-thalos:2007": "<p id=\"duncan_m:2021\">Duncan</p>",
    "berto:2006a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "streeter:2006": "<p id=\"duncan_m:2021\">Duncan</p>",
    "schnieder_b:2006d": "<p id=\"duncan_m:2021\">Duncan</p>",
    "fenner_dew:2006": "<p id=\"duncan_m:2021\">Duncan</p>",
    "kawall:2006a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "butt_d:2006": "<p id=\"duncan_m:2021\">Duncan</p>",
    "gert_j:2006": "<p id=\"duncan_m:2021\">Duncan</p>",
    "tanesini:2006": "<p id=\"duncan_m:2021\">Duncan</p>",
    "bernstein_m:2006": "<p id=\"duncan_m:2021\">Duncan</p>",
    "raterman:2006": "<p id=\"duncan_m:2021\">Duncan</p>",
    "moller_d:2006": "<p id=\"duncan_m:2021\">Duncan</p>",
    "mele_a:2006b": "<p id=\"duncan_m:2021\">Duncan</p>",
    "brogaard-salerno:2006": "<p id=\"duncan_m:2021\">Duncan</p>",
    "atkin:2006": "<p id=\"duncan_m:2021\">Duncan</p>",
    "mintoff:2006": "<p id=\"duncan_m:2021\">Duncan</p>",
    "schnieder_b:2006e": "<p id=\"duncan_m:2021\">Duncan</p>",
    "tiffany:2006a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "huemer_m:2006": "<p id=\"duncan_m:2021\">Duncan</p>",
    "jollimore:2006": "<p id=\"duncan_m:2021\">Duncan</p>",
    "oakes_ra:2006": "<p id=\"duncan_m:2021\">Duncan</p>",
    "vontevenar:2006": "<p id=\"duncan_m:2021\">Duncan</p>",
    "kallestrup:2006a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "corlett:2006": "<p id=\"duncan_m:2021\">Duncan</p>",
    "lang_g:2006": "<p id=\"duncan_m:2021\">Duncan</p>",
    "pollard_b:2006": "<p id=\"duncan_m:2021\">Duncan</p>",
    "marmodoro:2006": "<p id=\"duncan_m:2021\">Duncan</p>",
    "waller_bn:2006": "<p id=\"duncan_m:2021\">Duncan</p>",
    "brady_m:2006": "<p id=\"duncan_m:2021\">Duncan</p>",
    "hauser_k:2005": "<p id=\"duncan_m:2021\">Duncan</p>",
    "smith_ta:2005": "<p id=\"duncan_m:2021\">Duncan</p>",
    "silver_d:2005": "<p id=\"duncan_m:2021\">Duncan</p>",
    "balashov:2005b": "<p id=\"duncan_m:2021\">Duncan</p>",
    "taylor_js:2005a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "andreou:2005b": "<p id=\"duncan_m:2021\">Duncan</p>",
    "setiya:2005": "<p id=\"duncan_m:2021\">Duncan</p>",
    "nelson_mt:2005": "<p id=\"duncan_m:2021\">Duncan</p>",
    "mccormick_m:2005a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "lebar:2005a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "abell_c:2005a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "jenkins_cs:2005b": "<p id=\"duncan_m:2021\">Duncan</p>",
    "hetherington_s:2005": "<p id=\"duncan_m:2021\">Duncan</p>",
    "brown_ca-nagasawa:2005b": "<p id=\"duncan_m:2021\">Duncan</p>",
    "himma:2005": "<p id=\"duncan_m:2021\">Duncan</p>",
    "levy_k:2005a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "conly:2005": "<p id=\"duncan_m:2021\">Duncan</p>",
    "sommers_f:2005b": "<p id=\"duncan_m:2021\">Duncan</p>",
    "zangwill_n:2005": "<p id=\"duncan_m:2021\">Duncan</p>",
    "howell_rj:2005": "<p id=\"duncan_m:2021\">Duncan</p>",
    "crisp_tm:2005": "<p id=\"duncan_m:2021\">Duncan</p>",
    "rives:2005": "<p id=\"duncan_m:2021\">Duncan</p>",
    "andreou:2005a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "wray_kb:2005": "<p id=\"duncan_m:2021\">Duncan</p>",
    "walton_dn:2005a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "baumann_p:2005a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "goldman_ah:2004": "<p id=\"duncan_m:2021\">Duncan</p>",
    "tilley_jj:2004b": "<p id=\"duncan_m:2021\">Duncan</p>",
    "mcinerney_pk:2004": "<p id=\"duncan_m:2021\">Duncan</p>",
    "draper_p:2004": "<p id=\"duncan_m:2021\">Duncan</p>",
    "knapp_c:2004": "<p id=\"duncan_m:2021\">Duncan</p>",
    "caplan_b:2004": "<p id=\"duncan_m:2021\">Duncan</p>",
    "pettigrove:2004": "<p id=\"duncan_m:2021\">Duncan</p>",
    "carruthers:2004c": "<p id=\"duncan_m:2021\">Duncan</p>",
    "lemorvan:2004a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "richmond_am:2004a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "huenemann:2004a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "zimmerman_mj:2004": "<p id=\"duncan_m:2021\">Duncan</p>",
    "schechtman_m:2004": "<p id=\"duncan_m:2021\">Duncan</p>",
    "predelli:2004b": "<p id=\"duncan_m:2021\">Duncan</p>",
    "mendola:2004": "<p id=\"duncan_m:2021\">Duncan</p>",
    "radzik:2004": "<p id=\"duncan_m:2021\">Duncan</p>",
    "spicer_f:2004": "<p id=\"duncan_m:2021\">Duncan</p>",
    "bufacchi:2004": "<p id=\"duncan_m:2021\">Duncan</p>",
    "kriegel_u:2004b": "<p id=\"duncan_m:2021\">Duncan</p>",
    "becker_k:2004": "<p id=\"duncan_m:2021\">Duncan</p>",
    "graham_pj:2004": "<p id=\"duncan_m:2021\">Duncan</p>",
    "carter_a:2004": "<p id=\"duncan_m:2021\">Duncan</p>",
    "luper:2004b": "<p id=\"duncan_m:2021\">Duncan</p>",
    "hulse_d-etal:2004": "<p id=\"duncan_m:2021\">Duncan</p>",
    "wellman_c:2003": "<p id=\"duncan_m:2021\">Duncan</p>",
    "wall_e:2003": "<p id=\"duncan_m:2021\">Duncan</p>",
    "haji:2003a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "foran_s:2003": "<p id=\"duncan_m:2021\">Duncan</p>",
    "mason_e:2003": "<p id=\"duncan_m:2021\">Duncan</p>",
    "tomassi:2003": "<p id=\"duncan_m:2021\">Duncan</p>",
    "rescher:2003g": "<p id=\"duncan_m:2021\">Duncan</p>",
    "heil_j-robb:2003": "<p id=\"duncan_m:2021\">Duncan</p>",
    "bloomfield_p:2003": "<p id=\"duncan_m:2021\">Duncan</p>",
    "moore_ad:2003": "<p id=\"duncan_m:2021\">Duncan</p>",
    "hetherington_s:2003a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "kenyon_t:2003": "<p id=\"duncan_m:2021\">Duncan</p>",
    "holtman:2003": "<p id=\"duncan_m:2021\">Duncan</p>",
    "unwin_n:2003": "<p id=\"duncan_m:2021\">Duncan</p>",
    "smilansky:2003a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "hendrickson:2003": "<p id=\"duncan_m:2021\">Duncan</p>",
    "hills_a:2003a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "zangwill_n:2003": "<p id=\"duncan_m:2021\">Duncan</p>",
    "greenspan_ps:2003a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "kvanvig:2003b": "<p id=\"duncan_m:2021\">Duncan</p>",
    "hawley_k:2003a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "bruckner_dw:2003": "<p id=\"duncan_m:2021\">Duncan</p>",
    "mcgowan_mk:2003": "<p id=\"duncan_m:2021\">Duncan</p>",
    "douglas_h:2003": "<p id=\"duncan_m:2021\">Duncan</p>",
    "tiberius:2002a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "collins_jo:2002c": "<p id=\"duncan_m:2021\">Duncan</p>",
    "smith_nid:2002": "<p id=\"duncan_m:2021\">Duncan</p>",
    "goodman_c:2002": "<p id=\"duncan_m:2021\">Duncan</p>",
    "rajczi:2002a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "wall_s:2002": "<p id=\"duncan_m:2021\">Duncan</p>",
    "pritchard_d:2002a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "holbo:2002": "<p id=\"duncan_m:2021\">Duncan</p>",
    "sayward:2002b": "<p id=\"duncan_m:2021\">Duncan</p>",
    "becker_k:2002a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "schroer_r:2002a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "graham_go:2002": "<p id=\"duncan_m:2021\">Duncan</p>",
    "wisnewski_jj:2002": "<p id=\"duncan_m:2021\">Duncan</p>",
    "mele_a:2002": "<p id=\"duncan_m:2021\">Duncan</p>",
    "sorensen_ra:2002e": "<p id=\"duncan_m:2021\">Duncan</p>",
    "pendlebury_m:2002a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "dilworth_j:2002": "<p id=\"duncan_m:2021\">Duncan</p>",
    "elgin_cz:2002": "<p id=\"duncan_m:2021\">Duncan</p>",
    "helm_bw:2002": "<p id=\"duncan_m:2021\">Duncan</p>",
    "forrester_m:2002": "<p id=\"duncan_m:2021\">Duncan</p>",
    "stohr-wellman:2002": "<p id=\"duncan_m:2021\">Duncan</p>",
    "manion:2002": "<p id=\"duncan_m:2021\">Duncan</p>",
    "speak:2002": "<p id=\"duncan_m:2021\">Duncan</p>",
    "richmond_am:2001": "<p id=\"duncan_m:2021\">Duncan</p>",
    "thomasson:2001": "<p id=\"duncan_m:2021\">Duncan</p>",
    "conn_ch:2001": "<p id=\"duncan_m:2021\">Duncan</p>",
    "hetherington_s:2001b": "<p id=\"duncan_m:2021\">Duncan</p>",
    "portmore:2001": "<p id=\"duncan_m:2021\">Duncan</p>",
    "reimer_m:2001b": "<p id=\"duncan_m:2021\">Duncan</p>",
    "elder_c:2001c": "<p id=\"duncan_m:2021\">Duncan</p>",
    "savitt:2001": "<p id=\"duncan_m:2021\">Duncan</p>",
    "wellman_ca:2001": "<p id=\"duncan_m:2021\">Duncan</p>",
    "richardson_hes:2001": "<p id=\"duncan_m:2021\">Duncan</p>",
    "greenough_p:2001": "<p id=\"duncan_m:2021\">Duncan</p>",
    "metz_t:2001": "<p id=\"duncan_m:2021\">Duncan</p>",
    "harris_gw:2001": "<p id=\"duncan_m:2021\">Duncan</p>",
    "holland_s:2001": "<p id=\"duncan_m:2021\">Duncan</p>",
    "zemach:2001a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "scarre:2001a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "braennmark:2001": "<p id=\"duncan_m:2021\">Duncan</p>",
    "conee-feldman:2001": "<p id=\"duncan_m:2021\">Duncan</p>",
    "root_md:2001": "<p id=\"duncan_m:2021\">Duncan</p>",
    "mckenna_m:2001a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "arnold_dg:2001": "<p id=\"duncan_m:2021\">Duncan</p>",
    "suits_db:2001": "<p id=\"duncan_m:2021\">Duncan</p>",
    "craig_wl:2001c": "<p id=\"duncan_m:2021\">Duncan</p>",
    "mclean_gr:2001": "<p id=\"duncan_m:2021\">Duncan</p>",
    "hales_sd:2000a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "horowitz_a:2000": "<p id=\"duncan_m:2021\">Duncan</p>",
    "mulgan_t:2000": "<p id=\"duncan_m:2021\">Duncan</p>",
    "lango:2000a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "akiba:2000c": "<p id=\"duncan_m:2021\">Duncan</p>",
    "schellenberg_jl:2000": "<p id=\"duncan_m:2021\">Duncan</p>",
    "kristjansson_k:2000": "<p id=\"duncan_m:2021\">Duncan</p>",
    "forrester_m:2000": "<p id=\"duncan_m:2021\">Duncan</p>",
    "gert_j:2000": "<p id=\"duncan_m:2021\">Duncan</p>",
    "ebbs:2000": "<p id=\"duncan_m:2021\">Duncan</p>",
    "ridge:2000": "<p id=\"duncan_m:2021\">Duncan</p>",
    "davidson_m:2000": "<p id=\"duncan_m:2021\">Duncan</p>",
    "haldane_j:2000d": "<p id=\"duncan_m:2021\">Duncan</p>",
    "saka:2000": "<p id=\"duncan_m:2021\">Duncan</p>",
    "phillips_jo:2000": "<p id=\"duncan_m:2021\">Duncan</p>",
    "mcinerney_pk:2000": "<p id=\"duncan_m:2021\">Duncan</p>",
    "hales_sd:2000": "<p id=\"duncan_m:2021\">Duncan</p>",
    "richmond_am:2000a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "keller_s:2000": "<p id=\"duncan_m:2021\">Duncan</p>",
    "benatar:2000": "<p id=\"duncan_m:2021\">Duncan</p>",
    "helm_bw:2000": "<p id=\"duncan_m:2021\">Duncan</p>",
    "kekes:2000b": "<p id=\"duncan_m:2021\">Duncan</p>",
    "campbell_s:2000": "<p id=\"duncan_m:2021\">Duncan</p>",
    "pauen_m:2000": "<p id=\"duncan_m:2021\">Duncan</p>",
    "snow_ne:2000": "<p id=\"duncan_m:2021\">Duncan</p>",
    "elder_c:2000a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "price_tl:1999": "<p id=\"duncan_m:2021\">Duncan</p>",
    "cunningham_a:1999": "<p id=\"duncan_m:2021\">Duncan</p>",
    "francescotti:1999a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "sorensen_ra:1999e": "<p id=\"duncan_m:2021\">Duncan</p>",
    "weirich:1999": "<p id=\"duncan_m:2021\">Duncan</p>",
    "cappelen-winblad:1999": "<p id=\"duncan_m:2021\">Duncan</p>",
    "schwartz_sp:1999": "<p id=\"duncan_m:2021\">Duncan</p>",
    "jackman:1999": "<p id=\"duncan_m:2021\">Duncan</p>",
    "nicholson_g:1999": "<p id=\"duncan_m:2021\">Duncan</p>",
    "heil_j:1999": "<p id=\"duncan_m:2021\">Duncan</p>",
    "wellman_ca:1999": "<p id=\"duncan_m:2021\">Duncan</p>",
    "klein_cj:1999": "<p id=\"duncan_m:2021\">Duncan</p>",
    "davison_s:1999": "<p id=\"duncan_m:2021\">Duncan</p>",
    "millgram:1999": "<p id=\"duncan_m:2021\">Duncan</p>",
    "noe_a:1999": "<p id=\"duncan_m:2021\">Duncan</p>",
    "finkelstein_dh:1999": "<p id=\"duncan_m:2021\">Duncan</p>",
    "rowe_wl:1999": "<p id=\"duncan_m:2021\">Duncan</p>",
    "howardsnyder_d-howardsnyder:1999": "<p id=\"duncan_m:2021\">Duncan</p>",
    "haybron:1999": "<p id=\"duncan_m:2021\">Duncan</p>",
    "double:1999": "<p id=\"duncan_m:2021\">Duncan</p>",
    "kaufman_f:1999": "<p id=\"duncan_m:2021\">Duncan</p>",
    "sorensen_ra:1999d": "<p id=\"duncan_m:2021\">Duncan</p>",
    "radzik:1999": "<p id=\"duncan_m:2021\">Duncan</p>",
    "clarke_ste:1999": "<p id=\"duncan_m:2021\">Duncan</p>",
    "govier:1999": "<p id=\"duncan_m:2021\">Duncan</p>",
    "ridge:1998": "<p id=\"duncan_m:2021\">Duncan</p>",
    "mcinerney_pk:1998": "<p id=\"duncan_m:2021\">Duncan</p>",
    "tolhurst:1998": "<p id=\"duncan_m:2021\">Duncan</p>",
    "baker_r:1998": "<p id=\"duncan_m:2021\">Duncan</p>",
    "carter_wr-bahde:1998": "<p id=\"duncan_m:2021\">Duncan</p>",
    "umapathy:1998": "<p id=\"duncan_m:2021\">Duncan</p>",
    "heil_j:1998a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "mele_a:1997a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "sehon:1997": "<p id=\"duncan_m:2021\">Duncan</p>",
    "levine_jo:1997a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "rea_mc:1997a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "audi_r:1997h": "<p id=\"duncan_m:2021\">Duncan</p>",
    "burke_mb:1997": "<p id=\"duncan_m:2021\">Duncan</p>",
    "korcz:1997": "<p id=\"duncan_m:2021\">Duncan</p>",
    "koons_rc:1997a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "wright_ew:1996": "<p id=\"duncan_m:2021\">Duncan</p>",
    "mills_e:1996": "<p id=\"duncan_m:2021\">Duncan</p>",
    "moreland_jp:1996": "<p id=\"duncan_m:2021\">Duncan</p>",
    "sorensen_ra:1996a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "shalkowski:1996": "<p id=\"duncan_m:2021\">Duncan</p>",
    "code_l:1996": "<p id=\"duncan_m:2021\">Duncan</p>",
    "honderich:1995a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "anderson_dl:1995": "<p id=\"duncan_m:2021\">Duncan</p>",
    "sehon:1994": "<p id=\"duncan_m:2021\">Duncan</p>",
    "merricks:1994a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "oconnor_t:1994": "<p id=\"duncan_m:2021\">Duncan</p>",
    "machina_kf:1994": "<p id=\"duncan_m:2021\">Duncan</p>",
    "lenman:1994": "<p id=\"duncan_m:2021\">Duncan</p>",
    "carter_wr-hestevold:1994": "<p id=\"duncan_m:2021\">Duncan</p>",
    "tidman:1994": "<p id=\"duncan_m:2021\">Duncan</p>",
    "helm_bw:1994": "<p id=\"duncan_m:2021\">Duncan</p>",
    "murray_mj:1993": "<p id=\"duncan_m:2021\">Duncan</p>",
    "kolak:1993a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "jacquette:1993d": "<p id=\"duncan_m:2021\">Duncan</p>",
    "lafollette_h-shanks:1993": "<p id=\"duncan_m:2021\">Duncan</p>",
    "rosenberg_a:1992c": "<p id=\"duncan_m:2021\">Duncan</p>",
    "martin_raym:1992": "<p id=\"duncan_m:2021\">Duncan</p>",
    "senchuk:1991": "<p id=\"duncan_m:2021\">Duncan</p>",
    "heil_j-mele:1991": "<p id=\"duncan_m:2021\">Duncan</p>",
    "hale_s:1991a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "zimmerman_dw:1991": "<p id=\"duncan_m:2021\">Duncan</p>",
    "sartwell:1991": "<p id=\"duncan_m:2021\">Duncan</p>",
    "smith_ba:1991a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "church_j:1990": "<p id=\"duncan_m:2021\">Duncan</p>",
    "aquila:1990": "<p id=\"duncan_m:2021\">Duncan</p>",
    "haack_s:1990a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "eberle_r:1990": "<p id=\"duncan_m:2021\">Duncan</p>",
    "moreland_jp:1990": "<p id=\"duncan_m:2021\">Duncan</p>",
    "audi_r:1989a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "blumenfeld_d:1988": "<p id=\"duncan_m:2021\">Duncan</p>",
    "bonevac:1988": "<p id=\"duncan_m:2021\">Duncan</p>",
    "bunzl:1987": "<p id=\"duncan_m:2021\">Duncan</p>",
    "zimmerman_mj:1987": "<p id=\"duncan_m:2021\">Duncan</p>",
    "kolak-martin:1987": "<p id=\"duncan_m:2021\">Duncan</p>",
    "laudan_l:1987": "<p id=\"duncan_m:2021\">Duncan</p>",
    "garcia_j:1986": "<p id=\"duncan_m:2021\">Duncan</p>",
    "strawson_g:1986a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "garcia_jla:1986": "<p id=\"duncan_m:2021\">Duncan</p>",
    "graham_ge-stephens:1985": "<p id=\"duncan_m:2021\">Duncan</p>",
    "mcinerney_pk:1985": "<p id=\"duncan_m:2021\">Duncan</p>",
    "davis_wa:1984": "<p id=\"duncan_m:2021\">Duncan</p>",
    "hardin_cl:1984": "<p id=\"duncan_m:2021\">Duncan</p>",
    "odegard:1984": "<p id=\"duncan_m:2021\">Duncan</p>",
    "maloney_jc:1984": "<p id=\"duncan_m:2021\">Duncan</p>",
    "mccall_s:1984a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "nelson_jo:1983": "<p id=\"duncan_m:2021\">Duncan</p>",
    "morreall:1983": "<p id=\"duncan_m:2021\">Duncan</p>",
    "judge_b:1983": "<p id=\"duncan_m:2021\">Duncan</p>",
    "parsons_td:1982a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "prior_ew-etal:1982": "<p id=\"duncan_m:2021\">Duncan</p>",
    "watson_ra:1982": "<p id=\"duncan_m:2021\">Duncan</p>",
    "sober:1982": "<p id=\"duncan_m:2021\">Duncan</p>",
    "audi_r:1982c": "<p id=\"duncan_m:2021\">Duncan</p>",
    "davis_wa:1981": "<p id=\"duncan_m:2021\">Duncan</p>",
    "gaerdenfors:1981": "<p id=\"duncan_m:2021\">Duncan</p>",
    "sellars_w:1980b": "<p id=\"duncan_m:2021\">Duncan</p>",
    "lowe_ej:1980": "<p id=\"duncan_m:2021\">Duncan</p>",
    "creath:1980": "<p id=\"duncan_m:2021\">Duncan</p>",
    "nehamas:1979": "<p id=\"duncan_m:2021\">Duncan</p>",
    "kitcher_ph:1979": "<p id=\"duncan_m:2021\">Duncan</p>",
    "wilson_md:1979": "<p id=\"duncan_m:2021\">Duncan</p>",
    "robinson_ws:1979": "<p id=\"duncan_m:2021\">Duncan</p>",
    "rowe_wl:1979": "<p id=\"duncan_m:2021\">Duncan</p>",
    "foley_r:1979": "<p id=\"duncan_m:2021\">Duncan</p>",
    "king_jl:1979": "<p id=\"duncan_m:2021\">Duncan</p>",
    "gottlieb_dv:1978": "<p id=\"duncan_m:2021\">Duncan</p>",
    "foley_r:1978": "<p id=\"duncan_m:2021\">Duncan</p>",
    "lewis_dk:1978": "<p id=\"duncan_m:2021\">Duncan</p>",
    "adams_rm:1977": "<p id=\"duncan_m:2021\">Duncan</p>",
    "beversluis:1977": "<p id=\"duncan_m:2021\">Duncan</p>",
    "schlagel:1977": "<p id=\"duncan_m:2021\">Duncan</p>",
    "vaninwagen:1977": "<p id=\"duncan_m:2021\">Duncan</p>",
    "hartshorne_c:1977": "<p id=\"duncan_m:2021\">Duncan</p>",
    "davis_lh:1977": "<p id=\"duncan_m:2021\">Duncan</p>",
    "harman_gh:1977a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "cornman:1977": "<p id=\"duncan_m:2021\">Duncan</p>",
    "brand_m:1977": "<p id=\"duncan_m:2021\">Duncan</p>",
    "vanfraassen:1977a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "rawls:1977": "<p id=\"duncan_m:2021\">Duncan</p>",
    "jackson_f:1976": "<p id=\"duncan_m:2021\">Duncan</p>",
    "massey_gj:1976": "<p id=\"duncan_m:2021\">Duncan</p>",
    "suter_ro:1976": "<p id=\"duncan_m:2021\">Duncan</p>",
    "baier_ac:1976": "<p id=\"duncan_m:2021\">Duncan</p>",
    "alston:1976": "<p id=\"duncan_m:2021\">Duncan</p>",
    "oakley_it:1976": "<p id=\"duncan_m:2021\">Duncan</p>",
    "lewis_dk:1976a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "castaneda:1975b": "<p id=\"duncan_m:2021\">Duncan</p>",
    "rosenberg_a:1975": "<p id=\"duncan_m:2021\">Duncan</p>",
    "gean:1975": "<p id=\"duncan_m:2021\">Duncan</p>",
    "bricke:1975a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "nehamas:1975a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "sanford_dh:1975": "<p id=\"duncan_m:2021\">Duncan</p>",
    "audi_r:1974": "<p id=\"duncan_m:2021\">Duncan</p>",
    "gordon_rm:1974": "<p id=\"duncan_m:2021\">Duncan</p>",
    "swain_m:1974": "<p id=\"duncan_m:2021\">Duncan</p>",
    "sosa_e:1974": "<p id=\"duncan_m:2021\">Duncan</p>",
    "beversluis:1974a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "curley_em:1974": "<p id=\"duncan_m:2021\">Duncan</p>",
    "achinstein:1974a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "miller_b:1974a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "urmson_jo:1973": "<p id=\"duncan_m:2021\">Duncan</p>",
    "coder:1973": "<p id=\"duncan_m:2021\">Duncan</p>",
    "brownstein_d:1973a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "scarrow:1972": "<p id=\"duncan_m:2021\">Duncan</p>",
    "machina_kf:1972": "<p id=\"duncan_m:2021\">Duncan</p>",
    "cummins_r-gottlieb:1972": "<p id=\"duncan_m:2021\">Duncan</p>",
    "loux:1972": "<p id=\"duncan_m:2021\">Duncan</p>",
    "odegard:1972": "<p id=\"duncan_m:2021\">Duncan</p>",
    "george_r:1972": "<p id=\"duncan_m:2021\">Duncan</p>",
    "ziedins:1971": "<p id=\"duncan_m:2021\">Duncan</p>",
    "fisk:1971": "<p id=\"duncan_m:2021\">Duncan</p>",
    "alston:1971": "<p id=\"duncan_m:2021\">Duncan</p>",
    "adams_rm:1971a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "gruenbaum_a:1971a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "baier_ac:1971": "<p id=\"duncan_m:2021\">Duncan</p>",
    "perkins_m:1970": "<p id=\"duncan_m:2021\">Duncan</p>",
    "kyburg_he:1970b": "<p id=\"duncan_m:2021\">Duncan</p>",
    "wolterstorff:1970a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "shoemaker_ss:1970a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "bennett_j:1970": "<p id=\"duncan_m:2021\">Duncan</p>",
    "rorty_r:1970b": "<p id=\"duncan_m:2021\">Duncan</p>",
    "brandt_rb:1970": "<p id=\"duncan_m:2021\">Duncan</p>",
    "grossmann_rs:1969a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "eberle_r:1969a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "pike_n:1969": "<p id=\"duncan_m:2021\">Duncan</p>",
    "prior_an:1969d": "<p id=\"duncan_m:2021\">Duncan</p>",
    "castaneda:1969": "<p id=\"duncan_m:2021\">Duncan</p>",
    "spiegelberg:1968": "<p id=\"duncan_m:2021\">Duncan</p>",
    "goldman_ai:1968": "<p id=\"duncan_m:2021\">Duncan</p>",
    "saunders_jt:1968": "<p id=\"duncan_m:2021\">Duncan</p>",
    "cook_jw:1968": "<p id=\"duncan_m:2021\">Duncan</p>",
    "hintikka_j:1967d": "<p id=\"duncan_m:2021\">Duncan</p>",
    "vonwright_gh:1967c": "<p id=\"duncan_m:2021\">Duncan</p>",
    "ofstad_haro:1967": "<p id=\"duncan_m:2021\">Duncan</p>",
    "nakhnikian:1967a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "castaneda:1967h": "<p id=\"duncan_m:2021\">Duncan</p>",
    "fingarette:1967": "<p id=\"duncan_m:2021\">Duncan</p>",
    "kim_jw:1966": "<p id=\"duncan_m:2021\">Duncan</p>",
    "chisholm:1966b": "<p id=\"duncan_m:2021\">Duncan</p>",
    "mccall_s:1966b": "<p id=\"duncan_m:2021\">Duncan</p>",
    "sylvan_r-macrae:1966": "<p id=\"duncan_m:2021\">Duncan</p>",
    "scotttaggart_mj:1966": "<p id=\"duncan_m:2021\">Duncan</p>",
    "feinberg_j:1966": "<p id=\"duncan_m:2021\">Duncan</p>",
    "prior_an:1966": "<p id=\"duncan_m:2021\">Duncan</p>",
    "fitch_fb:1966": "<p id=\"duncan_m:2021\">Duncan</p>",
    "hochberg:1966a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "meiland:1966a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "bennett_j:1965a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "popkin_rh:1965": "<p id=\"duncan_m:2021\">Duncan</p>",
    "sellars_w:1965a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "danto:1965a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "campbell_kei:1965a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "mackie_jl:1965a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "kyburg_he:1965": "<p id=\"duncan_m:2021\">Duncan</p>",
    "chihara-fodor:1965": "<p id=\"duncan_m:2021\">Duncan</p>",
    "gentzen:1965": "<p id=\"duncan_m:2021\">Duncan</p>",
    "frankfurt:1965": "<p id=\"duncan_m:2021\">Duncan</p>",
    "achinstein:1965a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "nerlich_g:1965": "<p id=\"duncan_m:2021\">Duncan</p>",
    "kyburg_he:1964": "<p id=\"duncan_m:2021\">Duncan</p>",
    "hintikka_j:1964a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "levi_i-morgenbesser:1964": "<p id=\"duncan_m:2021\">Duncan</p>",
    "bracken_hm:1964": "<p id=\"duncan_m:2021\">Duncan</p>",
    "alston:1964a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "chisholm:1964": "<p id=\"duncan_m:2021\">Duncan</p>",
    "thalberg:1964a": "<p id=\"duncan_m:2021\">Duncan</p>",
    "spiegelberg:1964": "<p id=\"duncan_m:2021\">Duncan</p>",
    "taylor_rn:1964": "<p id=\"duncan_m:2021\">Duncan</p>",
    "sanford_dh:1959": "<p id=\"duncan_m:2021\">Duncan</p>",


}

main_bibkeys = frozenset((
    "hello:1919"
    "hello:1920",
    "hello:1921",
    "hello:1923",
    "hello:1924",
    "hello:1926",
    "hello:1927",
    "hello:1928",
    "hello:1940",
    "hello:1941",
    ))


In [17]:

def get_bibdivs(bibdiv_dict: dict[str, str], main_bibkeys: frozenset[str]) -> tuple[str, ...]:
    main_bibkeys_divs_nones = tuple(
        bibdiv_dict.get(bibkey, None) for bibkey in main_bibkeys
    )
    return tuple(
        div for div in main_bibkeys_divs_nones if div is not None
    )


In [18]:
def get_bibdivs_v2(bibdiv_dict: dict[str, str], main_bibkeys: frozenset[str]) -> tuple[str, ...]:
    return tuple(
        bibdiv_dict.get(bibkey, "") for bibkey in main_bibkeys if bibdiv_dict.get(bibkey, "") != ""
    )


In [19]:
def get_bibdivs_v3(bibdiv_dict: dict[str, str], main_bibkeys: frozenset[str]) -> tuple[str, ...]:
    return tuple(
        v for k, v in bibdiv_dict.items() if k in main_bibkeys
    )

In [20]:
# Benchmarking
from time import time
from random import choice, randint

def benchmark(func, *args, iterations=100000): # type: ignore
    start = time()
    for _ in range(iterations):
        func(*args)
    end = time()
    return end - start

In [21]:
# Benchmark the three functions

benchmark_results = {
    "get_bibdivs": benchmark(get_bibdivs, bibdiv_dict, main_bibkeys), # type: ignore
    "get_bibdivs_v2": benchmark(get_bibdivs_v2, bibdiv_dict, main_bibkeys), # type: ignore
    "get_bibdivs_v3": benchmark(get_bibdivs_v3, bibdiv_dict, main_bibkeys), # type: ignore
}

In [22]:
print("Benchmark results:")
for func_name, duration in benchmark_results.items():
    print(f"{func_name}: {duration:.6f} seconds")

Benchmark results:
get_bibdivs: 0.147581 seconds
get_bibdivs_v2: 0.108243 seconds
get_bibdivs_v3: 1.763796 seconds


In [23]:
from typing import Callable
from ref_pipe.models import Bibliography, TBibDivDict, TBibkey, HTMLIssue, HTMLVolume, THTMLCollapsible

type TGetBibkeyHTML = Callable[
    [TBibkey], str,
]

def generate_struct_html(
    struct: THTMLCollapsible,
    bibdiv_dict: TBibDivDict,
) -> str:

    main_html = f""

    for year in struct:

        year_html_contents = f""
        for year_content in year.contents:

            if isinstance(year_content, str):
                bibdiv = bibdiv_dict.get(year_content, None)
                if bibdiv is None:
                    lgr.warning(f"Bibkey {year_content} not found in bibdiv_dict.")
                    continue
                year_html_contents += bibdiv
                year_html_contents += "\n"

            elif isinstance(year_content, HTMLVolume):

                vol_html_contents = f""
                for vol_content in year_content.contents:

                    if isinstance(vol_content, str):
                        bibdiv = bibdiv_dict.get(vol_content, None)
                        if bibdiv is None:
                            lgr.warning(f"Bibkey {vol_content} not found in bibdiv_dict.")
                            continue
                        vol_html_contents += bibdiv
                        vol_html_contents += "\n"

                    elif isinstance(vol_content, HTMLIssue):
                        issue_html_contents = f""

                        for issue_content in vol_content.contents:
                            bibdiv = bibdiv_dict.get(issue_content, None)
                            if bibdiv is None:
                                lgr.warning(f"Bibkey {issue_content} not found in bibdiv_dict.")
                                continue
                            issue_html_contents += bibdiv
                            issue_html_contents += "\n"
                        
                        issue_html = issue_format.format(
                            name=vol_content.name,
                            contents=issue_html_contents
                        )
                        vol_html_contents += issue_html
                        vol_html_contents += "\n"
                    else:
                        raise ValueError(f"Unknown content type for volume content: {type(vol_content)}")

                volume_html = volume_format.format(
                    name=year_content.name,
                    contents=vol_html_contents
                )
                year_html_contents += volume_html
                year_html_contents += "\n"

            else:
                raise ValueError(f"Unknown content type for year content: {type(year_content)}")


        year_html = year_format.format(
            name=year.name,
            contents=year_html_contents
        )
        main_html += year_html
        main_html += "\n"

    return main_html



In [24]:
html = generate_struct_html(struct, bibdiv_dict)

2025-04-27 18:32:48 - __main__ - WARNING - Bibkey lowe_ej:1982c not found in bibdiv_dict.
2025-04-27 18:32:48 - __main__ - WARNING - Bibkey haugeland:1982 not found in bibdiv_dict.
2025-04-27 18:32:48 - __main__ - WARNING - Bibkey swinburne:1982a not found in bibdiv_dict.
2025-04-27 18:32:48 - __main__ - WARNING - Bibkey plantinga_a:1978 not found in bibdiv_dict.
2025-04-27 18:32:48 - __main__ - WARNING - Bibkey kim_jw:1978 not found in bibdiv_dict.
2025-04-27 18:32:48 - __main__ - WARNING - Bibkey dennett:1978k not found in bibdiv_dict.
2025-04-27 18:32:48 - __main__ - WARNING - Bibkey mcmullin_e:1978a not found in bibdiv_dict.
2025-04-27 18:32:48 - __main__ - WARNING - Bibkey chisholm:1978a not found in bibdiv_dict.


In [25]:

with open("apq.html", "w") as f:
    f.write(html)
    f.write("\n")